<a href="https://colab.research.google.com/github/ngminhtrung/models/blob/master/IBM_Docling_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here’s an updated minimal Colab notebook that:

Uploads a PDF

*   Converts it to Markdown with Docling
*   Captures performance + environment metadata
file size

pages

duration

pages/sec, MB/sec

Python version

Docling version

RAM

GPU availability and name

Exports metadata as a JSON file you can download

🔹 Cell 1 — Install dependencies

In [1]:
!pip install -q docling psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.5/273.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 105.9 MB/s eta 0:00:00
   ━

🔹 Cell 2 — Upload PDF

In [2]:
from google.colab import files

uploaded = files.upload()   # choose your PDF file
pdf_name = list(uploaded.keys())[0]
print("Uploaded file:", pdf_name)


Saving small-techcombank-vas-consolidated-financial-statements-interim25-searchable.pdf to small-techcombank-vas-consolidated-financial-statements-interim25-searchable.pdf
Uploaded file: small-techcombank-vas-consolidated-financial-statements-interim25-searchable.pdf


🔹 Cell 3 — Convert PDF → JSON and collect performance metadata

In [5]:
import os
import time
import json
import platform
import psutil

# Docling imports (no __version__ available)
from docling.document_converter import DocumentConverter

# Detect GPU availability (optional)
try:
    import torch
    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else None
except Exception:
    gpu_available = False
    gpu_name = None

# File size
file_size_bytes = os.path.getsize(pdf_name)
file_size_mb = file_size_bytes / (1024 * 1024)

# Convert with Docling
converter = DocumentConverter()

start = time.perf_counter()
result = converter.convert(pdf_name)
end = time.perf_counter()

duration_sec = end - start
doc = result.document
pages = len(doc.pages) if doc and doc.pages else 0

pages_per_sec = pages / duration_sec if duration_sec > 0 else 0.0
mb_per_sec = file_size_mb / duration_sec if duration_sec > 0 else 0.0

# Export JSON (Docling structured output)
json_obj = doc.export_to_dict()

json_file = pdf_name.rsplit(".", 1)[0] + "_docling.json"
with open(json_file, "w", encoding="utf-8") as f:
    json.dump(json_obj, f, indent=2, ensure_ascii=False)

# Collect metadata (without Docling version)
vm = psutil.virtual_memory()

metadata = {
    "file_name": pdf_name,
    "file_size_bytes": file_size_bytes,
    "file_size_mb": file_size_mb,
    "pages": pages,
    "duration_sec": duration_sec,
    "pages_per_sec": pages_per_sec,
    "mb_per_sec": mb_per_sec,
    "python_version": platform.python_version(),
    "python_build": platform.python_build(),
    "python_implementation": platform.python_implementation(),
    "platform": platform.platform(),
    "processor": platform.processor(),
    "gpu_available": gpu_available,
    "gpu_name": gpu_name,
    "total_ram_gb": vm.total / (1024**3),
    "available_ram_gb": vm.available / (1024**3),
}

meta_file = pdf_name.rsplit(".", 1)[0] + "_docling_metadata.json"
with open(meta_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("JSON file:", json_file)
print("Metadata file:", meta_file)
print("\n=== Summary ===")
for k, v in metadata.items():
    print(f"{k}: {v}")


[INFO] 2025-12-06 00:06:36,783 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-12-06 00:06:36,788 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.4.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-12-06 00:06:38,081 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2025-12-06 00:06:38,315 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-12-06 00:06:38,317 [RapidOCR] torch.py:54: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2025-12-06 00:06:38,789 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2025-12-06 00:06:38,791 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.4.0/torch/PP-OCRv4/cls/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2025-12-06 00:06:39,599 [RapidOCR] downlo

JSON file: small-techcombank-vas-consolidated-financial-statements-interim25-searchable_docling.json
Metadata file: small-techcombank-vas-consolidated-financial-statements-interim25-searchable_docling_metadata.json

=== Summary ===
file_name: small-techcombank-vas-consolidated-financial-statements-interim25-searchable.pdf
file_size_bytes: 1153773
file_size_mb: 1.1003236770629883
pages: 93
duration_sec: 104.71086668199996
pages_per_sec: 0.8881599679853182
mb_per_sec: 0.010508209051545711
python_version: 3.12.12
python_build: ('main', 'Oct 10 2025 08:52:57')
python_implementation: CPython
platform: Linux-6.6.105+-x86_64-with-glibc2.35
processor: x86_64
gpu_available: True
gpu_name: Tesla T4
total_ram_gb: 12.671436309814453
available_ram_gb: 9.139701843261719


🔹 Cell 4 — Download Markdown file

In [6]:
from google.colab import files

# Download the structured document JSON
files.download(json_file)

# Download the performance metadata JSON
files.download(meta_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>